# Phase 1: Preprocessing CICIDS2017 Dataset

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
RAW_DIR = "/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset"
OUT_DIR = "/kaggle/working/processed" 

In [ ]:
#a new directory will be craete where we will stor the output dataset after cleaning it.
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# STEP 1: Load and combine the CSV files
csv_paths = glob.glob(os.path.join(RAW_DIR, "*.csv"))
df_list = [pd.read_csv(path, low_memory=False, encoding="latin1") for path in csv_paths]
df = pd.concat(df_list)
df = df.reset_index(drop=True)
print("Combined shape:", df.shape)

In [ ]:
# STEP 2: Clean column names (strip stray whitespace)
df.columns = df.columns.str.strip()

In [ ]:
# STEP 3: Identify the label column
label_col = "Label"
print("Label column:", label_col)
print(df[label_col].value_counts())#gives the total number rows which contains that attacks 


In [ ]:
# STEP 4: Remove duplicate rows (before the split, to avoid the leakage)
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after dedup:", df.shape)

In [ ]:
# STEP 5: Handle infinite and missing values
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print("Rows with missing values:", df[numeric_cols].isna().any(axis=1).sum())
df = df.dropna(subset=numeric_cols)
print("Shape after dropping missing rows:", df.shape)

In [ ]:
# STEP 6: Drop identifier / leakage-risk columns becuse the 
#model will not learn from attack categories and will jsut memorise the dataset
id_cols = ["Flow ID", "Source IP", "Src IP", "Destination IP", "Dst IP", "Timestamp"]
df = df.drop(columns=[c for c in id_cols if c in df.columns])


In [ ]:
# STEP 7: Drop constant (zero-variance) columns because they contribute nothing to the training
numeric_cols = df.select_dtypes(include=[np.number]).columns  # recompute after drops
constant_cols = [c for c in numeric_cols if df[c].nunique() <= 1]
df = df.drop(columns=constant_cols)
print("Dropped constant columns:", constant_cols)

In [ ]:
# STEP 8: Build the labels
# 2 kinds of lables
# 1. binary label: benign=0,maliciou=1
# 2. attack category: the specific attack
df[label_col] = df[label_col].astype(str).str.strip()
df["Binary_Label"] = (df[label_col] != "BENIGN").astype(int)
df["Attack_Category"] = df[label_col]
df = df.drop(columns=[label_col])
print("\nClass distribution:")
print(df["Attack_Category"].value_counts())
#the below data shows each attack category occupies how many rows in the datset

In [ ]:
# STEP 9: Separate features/labels, then train/test split
feature_cols = [c for c in df.columns if c not in ("Binary_Label", "Attack_Category")]
X = df[feature_cols]
y = df["Binary_Label"]

X_train, X_test, y_train, y_test, cat_train, cat_test = train_test_split(
    X, y, df["Attack_Category"],
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])


In [ ]:
# STEP 10: Scale features (fit on train only, apply to test)
# we are scaling here becuase one column may be in millions and the other is represented using only number
#if there is a huge diffrence in the numbers then the model will get confused
#that is why we just scale to some similar avalues 
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

In [ ]:

# STEP 11: Save processed data
X_train_scaled["Binary_Label"] = y_train.values
X_train_scaled["Attack_Category"] = cat_train.values

X_test_scaled["Binary_Label"] = y_test.values
X_test_scaled["Attack_Category"] = cat_test.values

X_train_scaled.to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
X_test_scaled.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)

print("\nSaved train.csv and test.csv to", OUT_DIR)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

# Phase 2: EDA 

In [ ]:
#Exploratory traffic analysis : to knwo about how dat is distributed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


TRAIN_FILE = "/kaggle/working/processed/train.csv"


# Load the preprocessed training data
df = pd.read_csv(TRAIN_FILE)
print("Loaded data with shape:", df.shape)
print()

In [ ]:
# PART 1: Class distribution - how imbalanced is the data?
print("=" * 60)
print("PART 1: Class Distribution")
print("=" * 60)

# Count how many Normal (0) vs Attack (1) rows we have
binary_counts = df["Binary_Label"].value_counts()
print("\nNormal vs Attack counts:")
print(binary_counts)

# Show the same thing as a percentage, easier to understand imbalance
binary_percent = df["Binary_Label"].value_counts(normalize=True) * 100
print("\nNormal vs Attack percentage:")
print(binary_percent.round(2))

# Count each individual attack category too
print("\nCounts per attack category:")
print(df["Attack_Category"].value_counts())

# Make a simple bar chart of attack categories
plt.figure(figsize=(10, 5))
df["Attack_Category"].value_counts().plot(kind="bar")
plt.title("Number of Flows per Attack Category")
plt.ylabel("Number of flows")
plt.xlabel("Category")
plt.tight_layout()
plt.savefig("class_distribution.png")
plt.close()
print("\nSaved chart: class_distribution.png")
#from the output you can see that 83% is normal data and the other nearly 17% is attack category. this shows class imbalance

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/class_distribution.png"))

In [ ]:
  import os
  print(os.listdir("/kaggle/working"))

In [ ]:
# PART 2: Compare a few key features between Normal and Attack traffic
print("\n" + "=" * 60)
print("PART 2: Normal vs Attack - Feature Comparison")
print("=" * 60)

# Pick a small number of important features to look at.

features_to_check = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
]

# keep only the features that actually exist in this dataset
features_to_check = [f for f in features_to_check if f in df.columns]
print("\nFeatures being compared:", features_to_check)
# For each feature, print the average value for Normal vs Attack rows
for feature in features_to_check:
    normal_avg = df[df["Binary_Label"] == 0][feature].mean()
    attack_avg = df[df["Binary_Label"] == 1][feature].mean()
    print(f"\n{feature}:")
    print(f"   Normal average: {normal_avg:.4f}")
    print(f"   Attack average: {attack_avg:.4f}")

# Now make a boxplot for each feature, comparing Normal vs Attack side by side
for feature in features_to_check:
    plt.figure(figsize=(6, 4))
    df.boxplot(column=feature, by="Binary_Label")
    plt.title(f"{feature}: Normal (0) vs Attack (1)")
    plt.suptitle("")  # removes the default ugly subtitle
    plt.xlabel("Binary_Label (0 = Normal, 1 = Attack)")
    plt.ylabel(feature)
    plt.tight_layout()
    safe_name = feature.replace("/", "_").replace(" ", "_")
    plt.savefig(f"boxplot_{safe_name}.png")
    plt.close()

print("\nSaved one boxplot image per feature (boxplot_<feature>.png)")


In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/boxplot_Total_Fwd_Packets.png"))
display(Image("/kaggle/working/boxplot_Flow_Duration.png"))
display(Image("/kaggle/working/boxplot_Flow_Bytes_s.png"))
display(Image("/kaggle/working/boxplot_Total_Backward_Packets.png"))
display(Image("/kaggle/working/boxplot_Flow_Packets_s.png"))

In [ ]:
# PART 3: Compare features ACROSS different attack categories
print("\n" + "=" * 60)
print("PART 3: Comparing Attack Categories to Each Other")
print("=" * 60)

# We take the top 5 most common categories (this usually includes BENIGN).
top_categories = df["Attack_Category"].value_counts().head(6).index.tolist()
print("\nTop categories being compared:", top_categories)

df_top = df[df["Attack_Category"].isin(top_categories)]

for feature in features_to_check:
    plt.figure(figsize=(8, 5))
    df_top.boxplot(column=feature, by="Attack_Category", rot=45)
    plt.title(f"{feature} across top attack categories")
    plt.suptitle("")
    plt.ylabel(feature)
    plt.tight_layout()
    safe_name = feature.replace("/", "_").replace(" ", "_")
    plt.savefig(f"category_boxplot_{safe_name}.png")
    plt.close()

print("\nSaved one boxplot per feature comparing categories (category_boxplot_<feature>.png)")

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/category_boxplot_Flow_Bytes_s.png"))

In [ ]:
# PART 4: Correlation heatmap - find features that repeat the same info
print("\n" + "=" * 60)
print("PART 4: Correlation Between Features")
print("=" * 60)

# Only use numeric columns, and exclude the label columns
numeric_df = df.select_dtypes(include=[np.number])
numeric_df = numeric_df.drop(columns=["Binary_Label"], errors="ignore")

# To keep the heatmap readable, only look at the first 20 numeric columns
columns_to_plot = numeric_df.columns[:20]
correlation_matrix = numeric_df[columns_to_plot].corr()

plt.figure(figsize=(12, 10))
plt.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(columns_to_plot)), columns_to_plot, rotation=90)
plt.yticks(range(len(columns_to_plot)), columns_to_plot)
plt.title("Correlation Heatmap (first 20 numeric features)")
plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.close()
print("\nSaved chart: correlation_heatmap.png")

# Print out any pair of features that are VERY highly correlated (>0.9)
print("\nHighly correlated feature pairs (correlation > 0.9):")
already_shown = set()
for col1 in correlation_matrix.columns:
    for col2 in correlation_matrix.columns:
        if col1 != col2:
            pair = tuple(sorted([col1, col2]))
            corr_value = correlation_matrix.loc[col1, col2]
            if abs(corr_value) > 0.9 and pair not in already_shown:
                print(f"   {col1}  <->  {col2}   correlation = {corr_value:.2f}")
                already_shown.add(pair)


In [ ]:

# PART 5: Check if any feature separates Attack vs Normal TOO perfectly
print("\n" + "=" * 60)
print("PART 5: Checking for Suspiciously Perfect Features")
print("=" * 60)
print("If a feature separates Normal/Attack almost perfectly, investigate")
print("why - it might be a real signal, or a data collection artifact.\n")

for feature in features_to_check:
    normal_values = df[df["Binary_Label"] == 0][feature]
    attack_values = df[df["Binary_Label"] == 1][feature]

    normal_max = normal_values.max()
    normal_min = normal_values.min()
    attack_max = attack_values.max()
    attack_min = attack_values.min()

    # If the value ranges for Normal and Attack don't overlap at all,
    # this feature ALONE can perfectly tell them apart - worth investigating.
    no_overlap = (normal_max < attack_min) or (attack_max < normal_min)

    print(f"{feature}:")
    print(f"   Normal range: {normal_min:.2f} to {normal_max:.2f}")
    print(f"   Attack range: {attack_min:.2f} to {attack_max:.2f}")
    if no_overlap:
        print("   WARNING: ranges do not overlap at all - investigate this feature!")
    print()

print("Done with Phase 2. Check the saved .png images in this folder.")

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/correlation_heatmap.png"))

In [ ]:
import os
print(os.listdir("."))

# Phase 3: Training the models

In [ ]:
import pandas as pd
import numpy as np
import time
 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
 
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report,
)


 
TRAIN_FILE = "/kaggle/working/processed/train.csv"
TEST_FILE = "/kaggle/working/processed/test.csv"
 

In [ ]:
# STEP 1: Load the preprocessed train and test data
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
 
# STEP 2: Separate features (X) from the target (y)
# We only need Binary_Label for this phase - Attack_Category is for Phase 4
columns_to_drop = ["Binary_Label", "Attack_Category"]
 
X_train = train_df.drop(columns=columns_to_drop)
y_train = train_df["Binary_Label"]
 
X_test = test_df.drop(columns=columns_to_drop)
y_test = test_df["Binary_Label"]
 
print("Number of features used:", X_train.shape[1])

In [ ]:
#STEP 3: A helper function that trains one model and prints ALL the
# security-relevant metrics your project guide asks for.
def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test):
    print("\n" + "=" * 60)
    print("MODEL:", model_name)
    print("=" * 60)
 
    #  Train the model, and time how long it takes 
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
 
    # Predict on the test set, and time this too 
    start_time = time.time()
    y_pred = model.predict(X_test)
    prediction_time = time.time() - start_time
 
    # Get predicted probabilities (needed for ROC-AUC) 
    # predict_proba gives two columns: [probability of 0, probability of 1]
    # we only want the probability of class 1 (Attack)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
 
    #  Core metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
 
    #  Confusion matrix
    # This gives us: true negatives, false positives, false negatives, true positives
    cm = confusion_matrix(y_test, y_pred)
    true_negative, false_positive, false_negative, true_positive = cm.ravel()
 
    # False Positive Rate
    # FPR = how many Normal flows were wrongly flagged as attacks
    # out of ALL the actual Normal flows
    false_positive_rate = false_positive / (false_positive + true_negative)
 
    # Print everything 
    print(f"Training time:      {training_time:.2f} seconds")
    print(f"Prediction time:     {prediction_time:.4f} seconds")
    print(f"Precision:           {precision:.4f}")
    print(f"Recall:              {recall:.4f}  (this is your attack detection rate)")
    print(f"F1-score:            {f1:.4f}")
    print(f"False Positive Rate: {false_positive_rate:.4f}")
    print(f"ROC-AUC:             {roc_auc:.4f}")
 
    print("\nConfusion Matrix:")
    print("                  Predicted Normal   Predicted Attack")
    print(f"Actual Normal        {true_negative:>10}         {false_positive:>10}")
    print(f"Actual Attack        {false_negative:>10}         {true_positive:>10}")
 
    print("\nFull classification report:")
    print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"]))
 
    # return the results so we can compare models at the end
    return {
        "model_name": model_name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positive_rate": false_positive_rate,
        "roc_auc": roc_auc,
        "training_time": training_time,
        "prediction_time": prediction_time,
    }
 
 

In [ ]:
# STEP 4: Train and evaluate all three models
results = []
 
#  Model 1: Logistic Regression (simple baseline) 
log_reg_model = LogisticRegression(max_iter=1000)
results.append(
    train_and_evaluate(log_reg_model, "Logistic Regression", X_train, y_train, X_test, y_test)
)
 
# Model 2: Random Forest (tree-based baseline) 
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
results.append(
    train_and_evaluate(random_forest_model, "Random Forest", X_train, y_train, X_test, y_test)
)
 
# ---- Model 3: XGBoost (primary model) ----
xgboost_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)
results.append(
    train_and_evaluate(xgboost_model, "XGBoost", X_train, y_train, X_test, y_test)
)
 
 
# STEP 5: Put all results side by side so you can compare models easily
print("\n" + "=" * 60)
print("SUMMARY: All Models Compared")
print("=" * 60)
 
results_df = pd.DataFrame(results)
results_df = results_df.set_index("model_name")
print(results_df.round(4))
 
results_df.to_csv("phase3_model_comparison.csv")
print("\nSaved comparison table to: phase3_model_comparison.csv")
 
print("\nDone with Phase 3 baseline evaluation.")


# Phase 4: Multiclass Attack Classification

In [ ]:
import pandas as pd
import numpy as np
 
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from xgboost import XGBClassifier



TRAIN_FILE = "/kaggle/working/processed/train.csv"
TEST_FILE = "/kaggle/working/processed/test.csv"

 
# STEP 1: Load the preprocessed data
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
print("\nCategory counts in training data:")
category_counts = train_df["Attack_Category"].value_counts()
print(category_counts)
 
# Set a minimum sample threshold. Categories with fewer training rows than
# this get EXCLUDED from multiclass training (they cannot be reliably
# learned or evaluated with so few examples).
MIN_SAMPLES = 50

 
categories_to_keep = category_counts[category_counts >= MIN_SAMPLES].index.tolist()
categories_dropped = category_counts[category_counts < MIN_SAMPLES].index.tolist()
 
print(f"\nKeeping {len(categories_to_keep)} categories with >= {MIN_SAMPLES} samples")
print("Categories kept:", categories_to_keep)
print("Categories dropped (too rare for reliable multiclass modelling):", categories_dropped)

In [ ]:
train_filtered = train_df[train_df["Attack_Category"].isin(categories_to_keep)]
test_filtered = test_df[test_df["Attack_Category"].isin(categories_to_keep)]
 
print("\nTrain shape after filtering rare categories:", train_filtered.shape)
print("Test shape after filtering rare categories:", test_filtered.shape)
 
 
# STEP 4: Prepare features (X) and multiclass target (y)
columns_to_drop = ["Binary_Label", "Attack_Category"]
 
X_train = train_filtered.drop(columns=columns_to_drop)
X_test = test_filtered.drop(columns=columns_to_drop)
 
# XGBoost needs numeric labels, not text like "DDoS" or "BENIGN".
# LabelEncoder converts each category name into a number (0, 1, 2, ...)
label_encoder = LabelEncoder()
label_encoder.fit(train_filtered["Attack_Category"])
 
y_train = label_encoder.transform(train_filtered["Attack_Category"])
y_test = label_encoder.transform(test_filtered["Attack_Category"])
 
# Keep the mapping so we can turn numbers back into readable names later
class_names = label_encoder.classes_
print("\nClass number -> Category name mapping:")
for number, name in enumerate(class_names):
    print(f"   {number} -> {name}")
 
 
# STEP 5: Train the multiclass XGBoost model
print("\nTraining multiclass XGBoost model...")
 
multiclass_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1,
)
multiclass_model.fit(X_train, y_train)
 
y_pred = multiclass_model.predict(X_test)
print("Training complete.")

In [ ]:
#STEP 6: Multiclass confusion matrix
print("\n" + "=" * 60)
print("Multiclass Confusion Matrix")
print("=" * 60)
 
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df)
 
cm_df.to_csv("phase4_confusion_matrix.csv")
print("\nSaved confusion matrix to: phase4_confusion_matrix.csv")
"""
How to read this table:
Rows = the ACTUAL category
Columns = what the model PREDICTED
Numbers on the diagonal = correct predictions
Numbers off the diagonal = confusions (e.g. row=DDoS, column=BENIGN
means some real DDoS flows were predicted as BENIGN)
"""

# STEP 7: Per-class precision, recall, F1-score
print("\n" + "=" * 60)
print("Per-Class Performance")
print("=" * 60)
 
report_text = classification_report(y_test, y_pred, target_names=class_names)
print(report_text)
 
# Also save this as a proper table (easier to sort and inspect than plain text)
report_dict = classification_report(
    y_test, y_pred, target_names=class_names, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv("phase4_per_class_report.csv")
print("Saved per-class report to: phase4_per_class_report.csv")
 
 
# STEP 8: Highlight the best and worst detected categories
print("\n" + "=" * 60)
print("Best and Worst Detected Categories (by Recall)")
print("=" * 60)
 
# Only look at the actual class rows, not the summary rows
# (classification_report adds extra rows like "accuracy", "macro avg", etc.)
class_only_report = report_df.drop(
    index=[r for r in report_df.index if r not in class_names]
)
class_only_report = class_only_report.sort_values("recall", ascending=False)
 
print("\nBest detected categories (highest recall):")
print(class_only_report[["precision", "recall", "f1-score", "support"]].head(5))
 
print("\nWorst detected categories (lowest recall):")
print(class_only_report[["precision", "recall", "f1-score", "support"]].tail(5))
 
print("\nDone with Phase 4.")


In [ ]:
import pandas as pd
import numpy as np
import gc

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Check the number of samples in each class
class_counts = pd.Series(y_train).value_counts().sort_index()

distribution = pd.DataFrame({
    "Attack Category": label_encoder.classes_,
    "Training Samples": [
        class_counts.get(i, 0)
        for i in range(len(label_encoder.classes_))
    ]
})

print("Original training class distribution:")
print(distribution.to_string(index=False))

In [ ]:
# Target size for minority classes
TARGET_COUNT = 50000

# Find classes with fewer than 50,000 training samples
sampling_strategy = {
    class_id: TARGET_COUNT
    for class_id, count in class_counts.items()
    if count < TARGET_COUNT
}

print("Classes to oversample:")
for class_id, target_count in sampling_strategy.items():
    print(
        label_encoder.classes_[class_id],
        "->",
        target_count,
        "samples"
    )

# Use the same model settings for every experiment
def create_model():
    return XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1
    )


# Store overall results and individual class reports
results = []
class_reports = {}
confusion_matrices = {}


def evaluate_model(model, method_name):
    y_pred = model.predict(X_test)

    report = classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_))
    )

    results.append({
        "Method": method_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": report["macro avg"]["precision"],
        "Macro Recall": report["macro avg"]["recall"],
        "Macro F1": report["macro avg"]["f1-score"],
        "Weighted F1": report["weighted avg"]["f1-score"]
    })

    class_reports[method_name] = pd.DataFrame(report).T
    confusion_matrices[method_name] = pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    )

    print("\n", "=" * 60)
    print(method_name)
    print("=" * 60)
    print(classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    del y_pred, report, cm
    gc.collect()


# --------------------------------------------------
# EXPERIMENT 1: Original imbalanced training data
# --------------------------------------------------
print("\nExperiment 1: Original imbalanced data")

model = create_model()
model.fit(X_train, y_train)

evaluate_model(model, "Original")

del model
gc.collect()


# --------------------------------------------------
# EXPERIMENT 2: Class weighting
# --------------------------------------------------
print("\nExperiment 2: Class weighting")

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

model = create_model()
model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

evaluate_model(model, "Class Weighting")

del model, sample_weights
gc.collect()


# --------------------------------------------------
# EXPERIMENT 3: Random Oversampling
# --------------------------------------------------
print("\nExperiment 3: Random Oversampling")

ros = RandomOverSampler(
    sampling_strategy=sampling_strategy,
    random_state=42
)

X_ros, y_ros = ros.fit_resample(X_train, y_train)

print("Shape after Random Oversampling:", X_ros.shape)

model = create_model()
model.fit(X_ros, y_ros)

evaluate_model(model, "Random Oversampling")

del model, X_ros, y_ros, ros
gc.collect()


# --------------------------------------------------
# EXPERIMENT 4: SMOTE
# --------------------------------------------------
print("\nExperiment 4: SMOTE")

smote = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=42,
    k_neighbors=5
)

X_smote, y_smote = smote.fit_resample(X_train, y_train)

print("Shape after SMOTE:", X_smote.shape)

model = create_model()
model.fit(X_smote, y_smote)

evaluate_model(model, "SMOTE")

del model, X_smote, y_smote, smote
gc.collect()


print("\nAll four experiments completed.")

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Macro F1",
    ascending=False
)

print("Final Comparison:")
print(results_df.round(4).to_string(index=False))

results_df.to_csv(
    "/kaggle/working/imbalance_comparison.csv",
    index=False
)

print("\nSaved to imbalance_comparison.csv")

In [ ]:
best_method = results_df.iloc[0]["Method"]

print("Best method by Macro F1:", best_method)

print("\nPer-class results:")
print(
    class_reports[best_method][
        ["precision", "recall", "f1-score", "support"]
    ].round(4)
)

print("\nConfusion matrix:")
print(confusion_matrices[best_method])

# Feature selection:
using XGBoost model 

In [ ]:
# Separate features (X) and target (y)

columns_to_drop = ["Binary_Label", "Attack_Category"]
 
X_train = train_filtered.drop(columns=columns_to_drop)
X_test = test_filtered.drop(columns=columns_to_drop)
label_encoder = LabelEncoder()
label_encoder.fit(train_filtered["Attack_Category"])
 
y_train = label_encoder.transform(train_filtered["Attack_Category"])
y_test = label_encoder.transform(test_filtered["Attack_Category"])
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel

# X = input features
# y = target/label

# Create XGBoost model
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

# Train the model
model.fit(X_train, y_train)

# Select features based on importance
#let us say we have 80 features then XGBOost will give some importnace value to each one of them and we dont know which one to take
#and that is why we compute the median and if the imporatnce lies above this median then we are going to kepp that feature
selector = SelectFromModel(
    model,
    threshold="median",
    prefit=True
)

# Transform X
X_selected = selector.transform(X_train)

# Get names of selected features
selected_columns = X_train.columns[selector.get_support()]

print("Original number of features:", X_train.shape[1])
print("Selected number of features:", X_selected.shape[1])

print("\nSelected features:")
print(selected_columns.tolist())
#next print out the dropped features
dropped_features = X_train.columns.difference(selected_columns)
print("Number of dropped features: ",len(dropped_features))
print(dropped_features.tolist())

#next do the class imbalancing and then do feature selection beacsue feature slection with XG Boost depends on the distribution of data


In [ ]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
import pandas as pd
import gc

# Use the selected features from feature selection
X_train_selected = X_train[selected_columns]
X_test_selected = X_test[selected_columns]

print("Original number of features:", X_train.shape[1])
print("Selected number of features:", X_train_selected.shape[1])

# Oversample training data only
class_counts = pd.Series(y_train).value_counts()

sampling_strategy = {
    cls: 50000
    for cls, count in class_counts.items()
    if count < 50000
}

smote = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_selected, y_train
)

print("Training shape after SMOTE:", X_train_smote.shape)

# Train XGBoost
model_selected = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1
)

model_selected.fit(X_train_smote, y_train_smote)

# Evaluate on the original, unchanged test data
y_pred_selected = model_selected.predict(X_test_selected)

print("\nClassification Report - Selected Features")
print(classification_report(
    y_test,
    y_pred_selected,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

print("Accuracy:", accuracy_score(y_test, y_pred_selected))
print("Macro F1:", f1_score(
    y_test, y_pred_selected, average="macro"
))
print("Weighted F1:", f1_score(
    y_test, y_pred_selected, average="weighted"
))

# Release the large resampled arrays
del X_train_smote, y_train_smote
gc.collect()

In [ ]:
%whos

In [ ]:
!pip install shap

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np

print("SHAP version:", shap.__version__)

In [ ]:
# Take a small sample from the test set
X_sample = X_test_selected.sample(
    n=500,
    random_state=42
)

# Create SHAP explainer
explainer = shap.TreeExplainer(model_selected)

# Calculate SHAP values
shap_values = explainer.shap_values(X_sample)

print("SHAP values calculated successfully!")

In [ ]:
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=X_sample.columns
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=X_sample.columns,
    plot_type="bar"
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=X_sample.columns
)

In [ ]:
plt.savefig(
    "/kaggle/working/shap_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
# Derive thresholds from benign training data only

benign_train = train_df[train_df["Binary_Label"] == 0]

candidate_features = ["Flow Packets/s", "Flow Bytes/s"]

available_features = [
    col for col in candidate_features
    if col in benign_train.columns
]

rule_thresholds = {}

for col in available_features:
    values = pd.to_numeric(
        benign_train[col], errors="coerce"
    )

    values = values.replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if len(values) > 0:
        rule_thresholds[col] = float(
            values.quantile(0.99)
        )

print("Available features:", available_features)
print("Rule thresholds:", rule_thresholds)

In [ ]:
def evaluate_rules(flow):
    triggered = []

    # Rule 1: Unusually high traffic rate
    for feature, threshold in rule_thresholds.items():
        value = pd.to_numeric(
            pd.Series([flow.get(feature, np.nan)]),
            errors="coerce"
        ).iloc[0]

        if pd.notna(value) and value > threshold:
            triggered.append(
                feature + " above benign 99th percentile"
            )

    # Rule 2: Strong forward/backward packet imbalance
    fwd = flow.get("Total Fwd Packets", 0)
    bwd = flow.get("Total Backward Packets", 0)

    fwd = max(float(fwd or 0), 0)
    bwd = max(float(bwd or 0), 0)

    ratio = (fwd + 1) / (bwd + 1)

    if ratio >= 20 or ratio <= 1 / 20:
        triggered.append(
            "Strong forward/backward packet imbalance"
        )

    return triggered

In [ ]:
example_flow = X_test.iloc[0].to_dict()

print("Triggered rules:")
print(evaluate_rules(example_flow))

In [ ]:

# Check the rate features in the test data

for col in ["Flow Packets/s", "Flow Bytes/s"]:
    values = pd.to_numeric(X_test[col], errors="coerce")
    values = values.replace([np.inf, -np.inf], np.nan).dropna()

    print("\nFeature:", col)
    print(values.describe(percentiles=[0.50, 0.90, 0.95, 0.99]))
    print("Values above our threshold:",
          (values > rule_thresholds[col]).sum())
    print("Percentage flagged:",
          round((values > rule_thresholds[col]).mean() * 100, 2), "%")


In [ ]:

# Compare each rate rule with the actual test labels

y_true_binary = test_df["Binary_Label"].astype(int)

for col in ["Flow Packets/s", "Flow Bytes/s"]:
    values = pd.to_numeric(X_test[col], errors="coerce")
    values = values.replace([np.inf, -np.inf], np.nan)

    rule_alert = (values > rule_thresholds[col]).fillna(False).astype(int)

    tp = ((rule_alert == 1) & (y_true_binary == 1)).sum()
    fp = ((rule_alert == 1) & (y_true_binary == 0)).sum()
    fn = ((rule_alert == 0) & (y_true_binary == 1)).sum()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    print("\nRule:", col)
    print("True positives:", tp)
    print("False positives:", fp)
    print("False negatives:", fn)
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))


In [ ]:

# Check whether the features are in their original scale
check_features = [
    "Flow Packets/s",
    "Flow Bytes/s",
    "Total Fwd Packets",
    "Total Backward Packets"
]

for col in check_features:
    print("\n", col)
    print("Training data:")
    print(train_df[col].describe()[["min", "50%", "max"]])
    print("Test features:")
    print(X_test[col].describe()[["min", "50%", "max"]])


In [ ]:
print("Scaler:", type(scaler).__name__)
print("Scaler feature count:", scaler.n_features_in_)
print("X_test feature count:", X_test.shape[1])

print("\nFirst 5 scaler features:")
print(list(scaler.feature_names_in_[:5]))

print("\nFirst 5 X_test columns:")
print(list(X_test.columns[:5]))

print("\nColumn order matches:",
      list(scaler.feature_names_in_) == list(X_test.columns))

In [ ]:

# Convert scaled training and test features back to original values

X_train_original = pd.DataFrame(
    scaler.inverse_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_original = pd.DataFrame(
    scaler.inverse_transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Original-scale training features:")
print(X_train_original[
    ["Flow Packets/s", "Flow Bytes/s",
     "Total Fwd Packets", "Total Backward Packets"]
].describe())

print("\nOriginal-scale test features:")
print(X_test_original[
    ["Flow Packets/s", "Flow Bytes/s",
     "Total Fwd Packets", "Total Backward Packets"]
].describe())


In [ ]:
print("X_train_original:", X_train_original.shape)
print("X_test_original:", X_test_original.shape)

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)

print("\nTraining labels:")
print(train_df["Binary_Label"].value_counts())

print("\nTest labels:")
print(test_df["Binary_Label"].value_counts())

print("\nNegative rate values:")
for col in ["Flow Packets/s", "Flow Bytes/s"]:
    print(col)
    print("Training:", (X_train_original[col] < 0).sum())
    print("Testing:", (X_test_original[col] < 0).sum())

In [ ]:
print("Possible label variables:")

for name in ["y_train", "y_test", "y_train_binary", "y_test_binary",
             "y_train_bin", "y_test_bin"]:
    if name in globals():
        value = globals()[name]
        print(name, "shape:", getattr(value, "shape", None))

In [ ]:
print("X_train index:", X_train.index[:5].tolist())
print("X_test index:", X_test.index[:5].tolist())

print("train_df index:", train_df.index[:5].tolist())
print("test_df index:", test_df.index[:5].tolist())

In [ ]:
import numpy as np
import pandas as pd

# Select benign training rows only
benign_train_original = X_train_original.loc[
    np.asarray(y_train) == 0
]

# Features used for behavioural rules
candidate_features = [
    "Flow Packets/s",
    "Flow Bytes/s"
]

rule_thresholds = {}

for col in candidate_features:
    # Convert values to numeric
    values = pd.to_numeric(
        benign_train_original[col],
        errors="coerce"
    )

    # Remove invalid infinite and negative rate values
    values = values.replace([np.inf, -np.inf], np.nan)
    values = values[values >= 0].dropna()

    if len(values) > 0:
        rule_thresholds[col] = float(values.quantile(0.99))

print("Number of benign training flows:", len(benign_train_original))
print("Rule thresholds:", rule_thresholds)

In [ ]:
for col in ["Flow Packets/s", "Flow Bytes/s"]:
    values = pd.to_numeric(
        benign_train_original[col],
        errors="coerce"
    ).replace([np.inf, -np.inf], np.nan).dropna()

    values = values[values >= 0]

    print("\nFeature:", col)
    print("Minimum:", values.min())
    print("50th percentile:", values.quantile(0.50))
    print("90th percentile:", values.quantile(0.90))
    print("95th percentile:", values.quantile(0.95))
    print("99th percentile:", values.quantile(0.99))
    print("99.5th percentile:", values.quantile(0.995))
    print("99.9th percentile:", values.quantile(0.999))
    print("Maximum:", values.max())
    print("Unique values:", values.nunique())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

y_true = np.asarray(y_test).astype(int)

for col, threshold in rule_thresholds.items():

    values = pd.to_numeric(
        X_test_original[col],
        errors="coerce"
    ).to_numpy()

    # Invalid rates should not trigger the rule
    flagged = np.isfinite(values) & (values >= 0) & (values > threshold)

    # 1 = rule flags the flow as suspicious
    y_rule = flagged.astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_rule, labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )
    fpr = fp / (fp + tn) if fp + tn else 0

    print("\nRule:", col)
    print("Threshold:", threshold)
    print("Flows flagged:", int(flagged.sum()))
    print("TP:", tp, "FP:", fp)
    print("FN:", fn, "TN:", tn)
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1-score:", round(f1, 4))
    print("False positive rate:", round(fpr, 4))

In [ ]:
print("y_train distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\ny_test distribution:")
print(pd.Series(y_test).value_counts().sort_index())

print("\ntest_df label distribution:")
print(test_df["Binary_Label"].value_counts().sort_index())

print("\nFirst 20 y_test labels:")
print(np.asarray(y_test)[:20])

print("\nFirst 20 test_df labels:")
print(test_df["Binary_Label"].iloc[:20].to_list())

In [ ]:
import numpy as np

# Convert multiclass labels to binary labels
y_train_binary = (np.asarray(y_train) != 0).astype(int)
y_test_binary = (np.asarray(y_test) != 0).astype(int)

print("Binary training labels:")
print("Benign:", np.sum(y_train_binary == 0))
print("Attack:", np.sum(y_train_binary == 1))

print("\nBinary test labels:")
print("Benign:", np.sum(y_test_binary == 0))
print("Attack:", np.sum(y_test_binary == 1))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Use binary labels: 0 = benign, 1 = attack
y_true = (np.asarray(y_test) != 0).astype(int)

for col, threshold in rule_thresholds.items():

    values = pd.to_numeric(
        X_test_original[col],
        errors="coerce"
    ).to_numpy()

    # Flag flows exceeding the threshold
    flagged = (
        np.isfinite(values) &
        (values >= 0) &
        (values > threshold)
    )

    y_rule = flagged.astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_rule, labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    fpr = fp / (fp + tn) if fp + tn else 0

    print("\nRule:", col)
    print("Threshold:", threshold)
    print("Flows flagged:", int(flagged.sum()))
    print("TP:", tp, "FP:", fp)
    print("FN:", fn, "TN:", tn)
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1-score:", round(f1, 4))
    print("False positive rate:", round(fpr, 4))

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Correct binary test labels
y_true = (np.asarray(y_test) != 0).astype(int)

# Original packet counts
fwd = X_test_original["Total Fwd Packets"].to_numpy()
bwd = X_test_original["Total Backward Packets"].to_numpy()

# Avoid division by zero
ratio = (fwd + 1) / (bwd + 1)

# Flag strong imbalance in either direction
flagged = (
    np.isfinite(fwd) &
    np.isfinite(bwd) &
    (fwd >= 0) &
    (bwd >= 0) &
    ((ratio >= 20) | (ratio <= 1 / 20))
)

y_rule = flagged.astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_rule, labels=[0, 1]
).ravel()

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)
fpr = fp / (fp + tn) if fp + tn else 0

print("Flows flagged:", int(flagged.sum()))
print("TP:", tp, "FP:", fp)
print("FN:", fn, "TN:", tn)
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))
print("False positive rate:", round(fpr, 4))

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Predict using the scaled test features
y_pred = binary_xgb.predict(X_test)

# Correct binary ground-truth labels
y_true = (np.asarray(y_test) != 0).astype(int)

# Evaluation
tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("Binary XGBoost Results")
print("----------------------")
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Benign", "Attack"],
    zero_division=0
))

In [ ]:
for name, obj in globals().copy().items():
    if hasattr(obj, "predict") and hasattr(obj, "predict_proba"):
        print(name, "->", type(obj).__name__)

In [ ]:
for name in ["xgboost_model", "model", "model_selected", "multiclass_model"]:
    obj = globals().get(name)

    if obj is not None:
        print("\nModel:", name)
        print("Classes:", getattr(obj, "classes_", "Not available"))
        print("Number of classes:",
              len(obj.classes_) if hasattr(obj, "classes_") else "Unknown")

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Predict using the binary XGBoost model
y_pred = xgboost_model.predict(X_test)

# Convert multiclass labels into binary labels
# 0 = Benign, any non-zero class = Attack
y_true = (np.asarray(y_test) != 0).astype(int)

# Calculate confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("Binary XGBoost Results")
print("----------------------")
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Benign", "Attack"],
    zero_division=0
))

In [ ]:
import numpy as np
import pandas as pd

# Binary ground-truth labels
y_true = (np.asarray(y_test) != 0).astype(int)

# XGBoost predictions
y_pred = np.asarray(xgboost_model.predict(X_test)).astype(int)

# Find attacks missed by XGBoost
missed_mask = (y_true == 1) & (y_pred == 0)

print("Total missed attacks:", int(missed_mask.sum()))

# Inspect selected features for missed attacks
features_to_inspect = [
    "Flow Packets/s",
    "Flow Bytes/s",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow Duration"
]

available_features = [
    col for col in features_to_inspect
    if col in X_test_original.columns
]

missed_attacks = X_test_original.loc[
    missed_mask, available_features
]

print("\nMissed attack feature summary:")
print(missed_attacks.describe().T)

In [ ]:
# Separate the test flows into three groups
correct_attacks = (y_true == 1) & (y_pred == 1)
missed_attacks = (y_true == 1) & (y_pred == 0)
benign_flows = (y_true == 0)

features_to_inspect = [
    "Flow Packets/s",
    "Flow Bytes/s",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow Duration"
]

for group_name, mask in [
    ("Missed attacks", missed_attacks),
    ("Detected attacks", correct_attacks),
    ("Benign flows", benign_flows)
]:
    print("\n", group_name)
    print(
        X_test_original.loc[mask, features_to_inspect]
        .median()
        .round(3)
    )

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

y_true = (np.asarray(y_test) != 0).astype(int)
y_pred = np.asarray(xgboost_model.predict(X_test)).astype(int)

byte_rate = X_test_original["Flow Bytes/s"].to_numpy()
duration = X_test_original["Flow Duration"].to_numpy()

# Candidate rule: low byte rate + duration near 1,000,000
rule_flagged = (
    np.isfinite(byte_rate) &
    np.isfinite(duration) &
    (byte_rate >= 0) &
    (byte_rate <= 20) &
    (duration >= 900_000) &
    (duration <= 1_100_000)
)

# Evaluate the rule by itself
tn, fp, fn, tp = confusion_matrix(
    y_true, rule_flagged.astype(int), labels=[0, 1]
).ravel()

print("RULE-ONLY RESULTS")
print("Flows flagged:", int(rule_flagged.sum()))
print("TP:", tp, "FP:", fp)
print("FN:", fn, "TN:", tn)

# Check whether it catches attacks missed by XGBoost
missed_by_model = (y_true == 1) & (y_pred == 0)
caught_missed = missed_by_model & rule_flagged

print("\nHYBRID POTENTIAL")
print("Attacks missed by XGBoost:", int(missed_by_model.sum()))
print("Missed attacks caught by rule:", int(caught_missed.sum()))
print("Additional benign flows flagged:",
      int(((y_true == 0) & rule_flagged).sum()))

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = (np.asarray(y_test) != 0).astype(int)
y_pred = np.asarray(xgboost_model.predict(X_test)).astype(int)

byte_rate = X_test_original["Flow Bytes/s"].to_numpy()
duration = X_test_original["Flow Duration"].to_numpy()

rule_flagged = (
    np.isfinite(byte_rate) &
    np.isfinite(duration) &
    (byte_rate >= 0) &
    (byte_rate <= 20) &
    (duration >= 900_000) &
    (duration <= 1_100_000)
).astype(int)

# Hybrid: flag as attack if either method flags it
hybrid_pred = ((y_pred == 1) | (rule_flagged == 1)).astype(int)

for name, predictions in [
    ("XGBoost", y_pred),
    ("Hybrid", hybrid_pred)
]:
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    print("\n", name)
    print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)
    print("Precision:", round(precision, 5))
    print("Recall:", round(recall, 5))
    print("F1-score:", round(f1, 5))

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Original-scale training features
X_rule_train = X_train_original.copy()

# Binary labels: 0 = benign, 1 = attack
y_rule_train = (np.asarray(y_train) != 0).astype(int)

# Split into rule-training and validation sets
X_rule_fit, X_rule_val, y_rule_fit, y_rule_val = train_test_split(
    X_rule_train,
    y_rule_train,
    test_size=0.20,
    random_state=42,
    stratify=y_rule_train
)

print("Rule-training rows:", len(X_rule_fit))
print("Validation rows:", len(X_rule_val))

print("\nRule-training labels:")
print("Benign:", np.sum(y_rule_fit == 0))
print("Attack:", np.sum(y_rule_fit == 1))

print("\nValidation labels:")
print("Benign:", np.sum(y_rule_val == 0))
print("Attack:", np.sum(y_rule_val == 1))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone

# Binary labels
y_binary = (np.asarray(y_train) != 0).astype(int)

# Split scaled and original-scale features together
X_fit_scaled, X_val_scaled, X_fit_original, X_val_original, y_fit, y_val = train_test_split(
    X_train,
    X_train_original,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

# Create a separate model with the same settings
validation_model = clone(xgboost_model)

# Train only on the 80% training subset
validation_model.fit(X_fit_scaled, y_fit)

# Predict on the unseen validation subset
val_predictions = validation_model.predict(X_val_scaled)

print("Validation model trained successfully!")
print("Validation rows:", len(y_val))
print("Validation predictions:", len(val_predictions))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Actual validation labels and predicted labels
y_true = y_val
y_pred = val_predictions

# Calculate metrics
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_true, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

In [ ]:
import pandas as pd
import numpy as np

# Identify missed and detected attacks
missed_mask = (y_val == 1) & (val_predictions == 0)
detected_mask = (y_val == 1) & (val_predictions == 1)
benign_mask = (y_val == 0)

# Compare network-flow features
features_to_check = [
    "Flow Packets/s",
    "Flow Bytes/s",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow Duration"
]

comparison = pd.DataFrame({
    "Missed attacks": X_val_original.loc[
        X_val_original.index[missed_mask], features_to_check
    ].median(),
    "Detected attacks": X_val_original.loc[
        X_val_original.index[detected_mask], features_to_check
    ].median(),
    "Benign flows": X_val_original.loc[
        X_val_original.index[benign_mask], features_to_check
    ].median()
})

print("Number of missed attacks:", missed_mask.sum())
print("Number of detected attacks:", detected_mask.sum())

print("\nMedian feature values:")
print(comparison)

In [ ]:
# Candidate pattern: low byte rate and long duration
byte_rate = X_rule_fit["Flow Bytes/s"]
duration = X_rule_fit["Flow Duration"]

valid = (
    np.isfinite(byte_rate) &
    np.isfinite(duration) &
    (byte_rate >= 0) &
    (duration >= 0)
)

# Use fixed, interpretable ranges for initial exploration
low_bytes = byte_rate <= 20
long_duration = duration.between(900_000, 1_100_000)

candidate_rule = valid & low_bytes & long_duration

for label, name in [(0, "Benign"), (1, "Attack")]:
    group = y_rule_fit == label
    total = group.sum()
    matched = (group & candidate_rule).sum()

    print(f"{name} flows matching rule: {matched:,} / {total:,}")
    print(f"{name} match rate: {matched / total:.4%}\n")

In [ ]:
# Candidate rule on validation data
byte_rate = X_val_original["Flow Bytes/s"]
duration = X_val_original["Flow Duration"]

valid = (
    np.isfinite(byte_rate) &
    np.isfinite(duration) &
    (byte_rate >= 0) &
    (duration >= 0)
)

rule_predictions = (
    valid &
    (byte_rate <= 20) &
    duration.between(900_000, 1_100_000)
).astype(int)

# Actual attacks missed by XGBoost
missed_mask = (y_val == 1) & (val_predictions == 0)

# Additional attacks caught by the rule
additional_attacks = missed_mask & (rule_predictions == 1)

print("Attacks missed by XGBoost:", missed_mask.sum())
print("Missed attacks caught by rule:", additional_attacks.sum())
print("Missed attacks still not caught:",
      missed_mask.sum() - additional_attacks.sum())

print("\nTotal validation flows flagged by rule:",
      rule_predictions.sum())

print("Benign flows flagged by rule:",
      ((y_val == 0) & (rule_predictions == 1)).sum())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Hybrid: classify as attack if either method flags it
hybrid_predictions = (
    (val_predictions == 1) |
    (rule_predictions == 1)
).astype(int)

def evaluate(name, actual, predicted):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(actual, predicted))
    print("Precision:", precision_score(actual, predicted, zero_division=0))
    print("Recall:", recall_score(actual, predicted, zero_division=0))
    print("F1 Score:", f1_score(actual, predicted, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(actual, predicted))

evaluate("XGBoost Only", y_val, val_predictions)
evaluate("Hybrid System", y_val, hybrid_predictions)

In [ ]:

# Get the probability of an attack from the XGBoost model

val_probabilities = validation_model.predict_proba(X_val_scaled)[:, 1]

print("First 10 attack probabilities:")
print(val_probabilities[:10])

print("\nMinimum probability:", val_probabilities.min())
print("Maximum probability:", val_probabilities.max())
print("Mean probability:", val_probabilities.mean())


In [ ]:
from sklearn.base import clone

validation_model = clone(xgboost_model)

validation_model.fit(X_fit_scaled, y_fit)

print("Validation model trained successfully!")

In [ ]:
print("xgboost_model exists:", 'xgboost_model' in globals())
print("X_train exists:", 'X_train' in globals())
print("scaler exists:", 'scaler' in globals())
print("validation_model exists:", 'validation_model' in globals())

In [ ]:
import numpy as np

print("Probability distribution:")

print("Low probability (below 0.30):",
      np.sum(val_probabilities < 0.30))

print("Moderate probability (0.30 to below 0.70):",
      np.sum((val_probabilities >= 0.30) &
             (val_probabilities < 0.70)))

print("High probability (0.70 and above):",
      np.sum(val_probabilities >= 0.70))

print("Total validation flows:", len(val_probabilities))

In [ ]:
print("X_val_original exists:", 'X_val_original' in globals())
print("y_val exists:", 'y_val' in globals())
print("val_predictions exists:", 'val_predictions' in globals())

if 'X_val_original' in globals():
    print("\nValidation data shape:", X_val_original.shape)
    print("Available columns:")
    print(X_val_original.columns.tolist())

In [ ]:
import numpy as np

# Get the two original features
byte_rate = X_val_original["Flow Bytes/s"].to_numpy()
duration = X_val_original["Flow Duration"].to_numpy()

# Check for valid values
valid_values = (
    np.isfinite(byte_rate) &
    np.isfinite(duration) &
    (byte_rate >= 0) &
    (duration >= 0)
)

# Apply the candidate rule
rule_flags = (
    valid_values &
    (byte_rate <= 20) &
    (duration >= 900000) &
    (duration <= 1100000)
)

# Compare the rule with the actual validation labels
actual_labels = np.asarray(y_val)
rule_predictions = rule_flags.astype(int)

tp = np.sum((actual_labels == 1) & (rule_predictions == 1))
fp = np.sum((actual_labels == 0) & (rule_predictions == 1))
fn = np.sum((actual_labels == 1) & (rule_predictions == 0))
tn = np.sum((actual_labels == 0) & (rule_predictions == 0))

print("Rule Evaluation")
print("----------------")
print("Total flows flagged:", np.sum(rule_flags))
print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

print("\nPrecision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

In [ ]:
# Check the number of flows flagged by the candidate rule
# and the model's predictions for those same flows.

print("Rule-flagged flows:", int(rule_flags.sum()))

print("\nAmong rule-flagged flows:")
print(
    "XGBoost predicted attack:",
    int(np.sum(rule_flags & (val_predictions == 1)))
)

print(
    "XGBoost predicted benign:",
    int(np.sum(rule_flags & (val_predictions == 0)))
)

print("\nAmong flows NOT flagged by the rule:")
print(
    "XGBoost predicted attack:",
    int(np.sum((~rule_flags) & (val_predictions == 1)))
)

print(
    "XGBoost predicted benign:",
    int(np.sum((~rule_flags) & (val_predictions == 0)))
)

In [ ]:
import numpy as np
import pandas as pd

# Create a decision table
decision_labels = []

for i in range(len(val_probabilities)):

    probability = val_probabilities[i]
    rule_triggered = rule_flags[i]

    if probability >= 0.70:
        decision = "Malicious - model-driven"

    elif probability >= 0.30:
        if rule_triggered:
            decision = "Suspicious - investigate"
        else:
            decision = "Suspicious - investigate"

    else:
        if rule_triggered:
            decision = "Uncertain - manual review"
        else:
            decision = "Likely benign"

    decision_labels.append(decision)

# Summarize the decisions
decision_counts = pd.Series(decision_labels).value_counts()

print("Hybrid Decision Engine Results")
print("--------------------------------")

for decision, count in decision_counts.items():
    print(f"{decision}: {count}")

print("\nTotal flows processed:", len(decision_labels))

In [ ]:

# Compare hybrid decisions with the actual validation labels

results = pd.DataFrame({
    "Actual": np.where(np.asarray(y_val) == 1, "Attack", "Benign"),
    "Decision": decision_labels
})

print(pd.crosstab(
    results["Decision"],
    results["Actual"],
    margins=True
))


In [ ]:

# Treat malicious, suspicious, and uncertain flows as alerts

alert_labels = np.array([
    decision != "Likely benign"
    for decision in decision_labels
]).astype(int)

actual = np.asarray(y_val)

tp = np.sum((actual == 1) & (alert_labels == 1))
fp = np.sum((actual == 0) & (alert_labels == 1))
fn = np.sum((actual == 1) & (alert_labels == 0))
tn = np.sum((actual == 0) & (alert_labels == 0))

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("Hybrid Alert Evaluation")
print("-----------------------")
print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))


In [ ]:

# Compare the three approaches on the validation set

comparison = pd.DataFrame({
    "Approach": ["XGBoost only", "Rules only", "Current hybrid"],
    "Precision": [
        0.9968186,
        0.1271,
        0.9885
    ],
    "Recall": [
        0.9983115,
        0.0012,
        0.9997
    ],
    "F1-score": [
        0.9975645,
        0.0024,
        0.9941
    ],
    "False Positives": [
        217,
        570,
        793
    ],
    "False Negatives": [
        115,
        68025,
        21
    ]
})

print(comparison.to_string(index=False))


In [ ]:

# Inspect key network-flow features

features_to_check = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "SYN Flag Count",
    "RST Flag Count",
    "Destination Port"
]

print("Feature statistics for validation data:\n")
print(X_val_original[features_to_check].describe().round(2).to_string())


In [ ]:

import numpy as np

# Select benign flows from the training portion
y_fit_array = np.asarray(y_fit)

benign_mask = (y_fit_array == 0)
benign_train = X_fit_original.loc[benign_mask]

features = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow IAT Mean"
]

print("Benign training flows:", len(benign_train))
print("\nBenign feature percentiles:")

for feature in features:
    values = benign_train[feature].to_numpy(dtype=float)

    # Exclude non-finite and negative values
    values = values[np.isfinite(values) & (values >= 0)]

    print(f"\n{feature}")
    print("50th percentile:", np.percentile(values, 50))
    print("90th percentile:", np.percentile(values, 90))
    print("95th percentile:", np.percentile(values, 95))
    print("99th percentile:", np.percentile(values, 99))


In [ ]:

import numpy as np
import pandas as pd

# Thresholds derived from benign training flows
packet_rate_threshold = 999999.9999999998
byte_rate_threshold = 12333333.333333299
fwd_packets_threshold = 58.99999999999994
bwd_packets_threshold = 73.99999999999997

Xv = X_val_original
actual = np.asarray(y_val)

# Get feature values
packet_rate = Xv["Flow Packets/s"].to_numpy(dtype=float)
byte_rate = Xv["Flow Bytes/s"].to_numpy(dtype=float)
fwd_packets = Xv["Total Fwd Packets"].to_numpy(dtype=float)
bwd_packets = Xv["Total Backward Packets"].to_numpy(dtype=float)

# Build the candidate rules
rules = {
    "High packet rate": (
        np.isfinite(packet_rate) &
        (packet_rate >= 0) &
        (packet_rate >= packet_rate_threshold)
    ),
    "High byte rate": (
        np.isfinite(byte_rate) &
        (byte_rate >= 0) &
        (byte_rate >= byte_rate_threshold)
    ),
    "High packet count": (
        np.isfinite(fwd_packets) &
        np.isfinite(bwd_packets) &
        (fwd_packets >= 0) &
        (bwd_packets >= 0) &
        (
            (fwd_packets >= fwd_packets_threshold) |
            (bwd_packets >= bwd_packets_threshold)
        )
    )
}

rows = []

for name, flags in rules.items():
    tp = np.sum(flags & (actual == 1))
    fp = np.sum(flags & (actual == 0))
    fn = np.sum((~flags) & (actual == 1))
    tn = np.sum((~flags) & (actual == 0))

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    rows.append({
        "Rule": name,
        "Flagged": int(flags.sum()),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4)
    })

print(pd.DataFrame(rows).to_string(index=False))


In [ ]:

# Inspect the distribution of validation labels

actual_labels = np.asarray(y_val)

label_counts = pd.Series(actual_labels).value_counts().sort_index()

print("Validation label distribution:")
print(label_counts.to_string())


In [ ]:
print("Available label-related variables:")

for name in globals():
    if any(word in name.lower() for word in ["label", "class", "target", "attack"]):
        value = globals()[name]

        if hasattr(value, "shape"):
            print(name, "shape:", value.shape)
        elif isinstance(value, (dict, list, tuple)):
            print(name, "type:", type(value).__name__, "length:", len(value))

In [ ]:
print("Class names:")
print(class_names)

print("\nOriginal training labels:")
print("Shape:", np.asarray(y_train).shape)
print("Unique labels:", np.unique(np.asarray(y_train)))

print("\nAttack values:")
print("Shape:", attack_values.shape)
print("First 10 values:", attack_values[:10])

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

y_multiclass_train = np.asarray(y_train)

_, _, _, _, _, _, _, y_multiclass_val = train_test_split(
    X_train,
    X_train_original,
    y_binary,
    y_multiclass_train,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print("Validation attack categories:")
print(
    pd.Series(y_multiclass_val)
    .map(dict(enumerate(class_names)))
    .value_counts()
    .to_string()
)

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

# Convert binary predictions back to an array
predictions = np.asarray(val_predictions)

# Actual multiclass labels for validation
actual_multiclass = np.asarray(y_multiclass_val)

print("Recall for each attack category:\n")

for class_id, class_name in enumerate(class_names):
    mask = actual_multiclass == class_id

    if mask.sum() > 0:
        if class_id == 0:
            # Benign traffic is correctly classified when prediction is 0
            correct = np.sum(predictions[mask] == 0)
        else:
            # Binary model detects an attack when prediction is 1
            correct = np.sum(predictions[mask] == 1)

        recall = correct / mask.sum()

        print(
            f"{class_name:30s} "
            f"Total: {mask.sum():6d}  "
            f"Detected: {correct:6d}  "
            f"Recall: {recall:.4f}"
        )

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith((".pcap", ".pcapng")):
            print(os.path.join(root, file))

In [ ]:

# Test the trained XGBoost model

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Replace xgb_model if your model has a different variable name
model = xgb_model

# Make predictions on validation data
y_pred = model.predict(X_val_scaled)

# Calculate evaluation metrics
print("XGBoost Model Evaluation")
print("-" * 40)

print("Accuracy :", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred, zero_division=0))
print("Recall   :", recall_score(y_val, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_val, y_pred, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)

print("\nConfusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

# False positive rate
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
print("False Positive Rate:", fpr)

# ROC-AUC
if hasattr(model, "predict_proba"):
    y_prob = model.predict_proba(X_val_scaled)[:, 1]
    print("ROC-AUC:", roc_auc_score(y_val, y_prob))


In [ ]:
import os

folder = "/kaggle/working/ids_artifacts"

if os.path.exists(folder):
    print(os.listdir(folder))
else:
    print("ids_artifacts folder not found")

In [ ]:

# CHECK ALL IMPORTANT MULTICLASS VARIABLES

print("=" * 60)
print("CHECKING EXISTING NOTEBOOK VARIABLES")
print("=" * 60)

# 1. Check trained models
print("\n1. POSSIBLE TRAINED MODELS:")

for name, obj in list(globals().items()):
    if any(word in name.lower() for word in ["model", "xgb", "classifier", "clf"]):
        if hasattr(obj, "predict"):
            print(f"  {name}: {type(obj).__name__}")

# 2. Check label encoders
print("\n2. LABEL ENCODERS:")

for name, obj in list(globals().items()):
    if "encoder" in name.lower() or "label_enc" in name.lower():
        print(f"  {name}: {type(obj).__name__}")

# 3. Check training and testing features
print("\n3. FEATURE DATAFRAMES:")

for name, obj in list(globals().items()):
    if any(word in name.lower() for word in
           ["x_train", "x_test", "xtrain", "xtest", "feature"]):
        if hasattr(obj, "shape") and hasattr(obj, "columns"):
            print(f"  {name}: shape={obj.shape}")

# 4. Check target variables
print("\n4. TARGET VARIABLES:")

for name, obj in list(globals().items()):
    if any(word in name.lower() for word in
           ["y_train", "y_test", "ytrain", "ytest", "target"]):
        if hasattr(obj, "__len__") and not isinstance(obj, str):
            try:
                print(f"  {name}: length={len(obj)}")
            except Exception:
                pass

# 5. Check multiclass-related variables
print("\n5. ALL MULTICLASS-RELATED VARIABLES:")

for name in list(globals().keys()):
    if any(word in name.lower() for word in
           ["multi", "smote", "selected", "shap", "label_encoder"]):
        print(f"  {name}")

print("\n" + "=" * 60)
print("CHECK COMPLETE")
print("=" * 60)

In [ ]:
# STEP 1: CHECK MULTICLASS MODEL

print("MODEL:", type(multiclass_model).__name__)
print("MODEL PARAMETERS:")

print("Objective:", multiclass_model.get_params().get("objective"))
print("Number of classes:", multiclass_model.get_params().get("num_class"))
print("Number of features:", multiclass_model.n_features_in_)

print("\nLABEL ENCODER CLASSES:")
print(label_encoder.classes_)
print("Total classes:", len(label_encoder.classes_))

print("\nSELECTED FEATURES:")
print("Training shape:", X_train_selected.shape)
print("Testing shape:", X_test_selected.shape)
print("Number of selected features:", len(selected_columns))

print("\nEXISTING PREDICTION:")
print("y_pred_selected exists:", "y_pred_selected" in globals())
print("actual_multiclass exists:", "actual_multiclass" in globals())

print("\nCHECK COMPLETE")

In [ ]:
# STEP 2: MULTICLASS MODEL EVALUATION

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
import matplotlib.pyplot as plt

# Predict using the existing multiclass model
y_pred_final = multiclass_model.predict(X_test_selected)

# Convert actual labels to a NumPy array
import numpy as np

y_actual_final = np.asarray(actual_multiclass)

# Display metrics
print("MULTICLASS XGBOOST RESULTS")
print("=" * 50)

print("Accuracy:", accuracy_score(y_actual_final, y_pred_final))

print("Macro F1:", f1_score(
    y_actual_final,
    y_pred_final,
    average="macro"
))

print("\nCLASSIFICATION REPORT:\n")

print(classification_report(
    y_actual_final,
    y_pred_final,
    zero_division=0
))

print("\nCONFUSION MATRIX:\n")

print(confusion_matrix(
    y_actual_final,
    y_pred_final
))

In [ ]:
# CHECK WHICH EXISTING MODEL MATCHES THE SELECTED FEATURES

models_to_check = {
    "multiclass_model": multiclass_model,
    "model_selected": model_selected,
    "model": model,
    "xgboost_model": xgboost_model
}

print("=" * 60)
print("MODEL AND FEATURE COMPATIBILITY CHECK")
print("=" * 60)

print("\nSelected training features:", X_train_selected.shape[1])
print("Selected testing features:", X_test_selected.shape[1])

for name, clf in models_to_check.items():
    try:
        print(f"\nModel: {name}")
        print("Expected features:", clf.n_features_in_)

        # Compare the model's feature names when available
        booster = clf.get_booster()
        model_features = booster.feature_names

        if model_features is not None:
            print("Stored feature names:", len(model_features))
            print(
                "Matches selected feature names:",
                list(model_features) == list(X_test_selected.columns)
            )

        if clf.n_features_in_ == X_test_selected.shape[1]:
            print("FEATURE COUNT MATCHES")
        else:
            print("FEATURE COUNT DOES NOT MATCH")

    except Exception as e:
        print("Could not inspect model:", e)

print("\nDone.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
import numpy as np
import matplotlib.pyplot as plt

# Use the model trained on the 35 selected features
y_pred_final = model_selected.predict(X_test_selected)

# Check the actual test labels
y_actual_final = np.asarray(actual_multiclass)

print("Prediction shape:", y_pred_final.shape)
print("Actual labels shape:", y_actual_final.shape)

# Ensure predictions and labels correspond to the same test rows
assert len(y_pred_final) == len(y_actual_final), (
    "Mismatch: predictions and actual labels have different lengths."
)

print("\n" + "=" * 60)
print("MULTICLASS XGBOOST EVALUATION")
print("=" * 60)

print(f"Accuracy: {accuracy_score(y_actual_final, y_pred_final):.4f}")
print(f"Macro F1-score: {f1_score(y_actual_final, y_pred_final, average='macro', zero_division=0):.4f}")

print("\nCLASSIFICATION REPORT:")
print(
    classification_report(
        y_actual_final,
        y_pred_final,
        zero_division=0
    )
)

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_actual_final, y_pred_final))

In [ ]:
import numpy as np
import pandas as pd

print("=" * 60)
print("CHECKING MULTICLASS TARGET VARIABLES")
print("=" * 60)

target_names = [
    "actual_multiclass",
    "y_test",
    "y_test_binary",
    "y_multiclass_train",
    "y_multiclass_val"
]

for name in target_names:
    if name in globals():
        value = globals()[name]
        print(f"{name}: shape/length = {np.asarray(value).shape}")

print("\n" + "=" * 60)
print("CHECKING FEATURE DATASETS")
print("=" * 60)

feature_names = [
    "X_train",
    "X_test",
    "X_train_selected",
    "X_test_selected",
    "X_train_original",
    "X_test_original"
]

for name in feature_names:
    if name in globals():
        value = globals()[name]
        print(f"{name}: shape = {value.shape}")

print("\n" + "=" * 60)
print("MODEL CLASS INFORMATION")
print("=" * 60)

print("Model:", type(model_selected).__name__)
print("Number of model features:", model_selected.n_features_in_)

if hasattr(model_selected, "classes_"):
    print("Model classes:", model_selected.classes_)

print("\nLabel encoder classes:")
print(label_encoder.classes_)

In [ ]:
import numpy as np
import pandas as pd

print("Unique values in y_test:")
print(np.unique(y_test))

print("\nUnique values in y_test_binary:")
print(np.unique(y_test_binary))

print("\nUnique values in y_multiclass_train:")
print(np.unique(y_multiclass_train))

print("\nUnique values in actual_multiclass:")
print(np.unique(actual_multiclass))

print("\nUnique values in model_selected classes:")
print(model_selected.classes_)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Predict using the existing trained model
y_pred_final = model_selected.predict(X_test_selected)

# Use the matching multiclass test labels
y_actual_final = np.asarray(y_test)

# Verify compatibility
assert len(y_pred_final) == len(y_actual_final)
assert np.array_equal(np.unique(y_actual_final), model_selected.classes_)

# Calculate metrics
accuracy = accuracy_score(y_actual_final, y_pred_final)
macro_f1 = f1_score(
    y_actual_final, y_pred_final,
    average="macro",
    zero_division=0
)
weighted_f1 = f1_score(
    y_actual_final, y_pred_final,
    average="weighted",
    zero_division=0
)

print("=" * 60)
print("MULTICLASS XGBOOST MODEL EVALUATION")
print("=" * 60)
print(f"Number of test samples: {len(y_actual_final):,}")
print(f"Number of classes: {len(model_selected.classes_)}")
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro F1-score: {macro_f1:.4f}")
print(f"Weighted F1-score: {weighted_f1:.4f}")

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    y_actual_final,
    y_pred_final,
    labels=model_selected.classes_,
    target_names=label_encoder.classes_,
    zero_division=0
))

# Plot confusion matrix
cm = confusion_matrix(
    y_actual_final,
    y_pred_final,
    labels=model_selected.classes_
)

plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("XGBoost Multiclass Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

# Use a manageable sample to keep SHAP computation practical
sample_size = min(1000, len(X_test_selected))

X_shap = X_test_selected.sample(
    n=sample_size,
    random_state=42
)

print("SHAP sample shape:", X_shap.shape)

# Create SHAP explainer for the existing XGBoost model
explainer = shap.TreeExplainer(model_selected)

# Calculate SHAP values
shap_values = explainer.shap_values(X_shap)

print("SHAP values calculated successfully!")

In [ ]:
# Summary plot of feature importance
shap.summary_plot(
    shap_values,
    X_shap,
    max_display=20,
    show=True
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Handle common SHAP output formats for multiclass models
if isinstance(shap_values, list):
    # Older SHAP versions: list containing one array per class
    shap_array = np.stack(shap_values, axis=0)

else:
    shap_array = np.asarray(shap_values)

print("SHAP array shape:", shap_array.shape)

# Expected shapes:
# (samples, features, classes)
# or (classes, samples, features)

if shap_array.ndim == 3:
    if shap_array.shape[0] == len(model_selected.classes_):
        # Format: (classes, samples, features)
        mean_abs_shap = np.abs(shap_array).mean(axis=(0, 1))

    elif shap_array.shape[1] == X_shap.shape[1]:
        # Format: (samples, features, classes)
        mean_abs_shap = np.abs(shap_array).mean(axis=(0, 2))

    else:
        raise ValueError(
            f"Unexpected SHAP dimensions: {shap_array.shape}"
        )

elif shap_array.ndim == 2:
    # Format: (samples, features)
    mean_abs_shap = np.abs(shap_array).mean(axis=0)

else:
    raise ValueError(
        f"Unexpected SHAP dimensions: {shap_array.shape}"
    )

# Build and display the importance table
shap_importance = pd.DataFrame({
    "Feature": X_shap.columns,
    "Mean_Absolute_SHAP": mean_abs_shap
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

print("\nTOP 15 FEATURES BY GLOBAL SHAP IMPORTANCE")
display(shap_importance.head(15))

# Plot top 15 features
top_features = shap_importance.head(15).sort_values(
    "Mean_Absolute_SHAP"
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_features["Feature"],
    top_features["Mean_Absolute_SHAP"]
)
plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Feature")
plt.title("Global SHAP Feature Importance — XGBoost")
plt.tight_layout()
plt.show()

In [ ]:
import os

os.makedirs("ids_artifacts", exist_ok=True)

# Save feature importance as a CSV file
shap_importance.to_csv(
    "ids_artifacts/shap_feature_importance.csv",
    index=False
)

print("SHAP feature importance saved successfully!")

In [ ]:
import os
import joblib

os.makedirs("ids_artifacts", exist_ok=True)

# Save the trained 12-class model
joblib.dump(
    model_selected,
    "ids_artifacts/xgboost_multiclass.pkl"
)

# Save the class-label encoder
joblib.dump(
    label_encoder,
    "ids_artifacts/label_encoder.pkl"
)

# Save the exact 35 features used by the model
joblib.dump(
    list(X_train_selected.columns),
    "ids_artifacts/selected_features.pkl"
)

print("All model artifacts saved successfully!")
print("Number of selected features:", len(X_train_selected.columns))
print("Number of classes:", len(model_selected.classes_))
print("Classes:", list(label_encoder.classes_))

In [ ]:
import os

print("Saved artifacts:")
for filename in os.listdir("ids_artifacts"):
    print(filename)

In [ ]:
%pip install streamlit

In [ ]:
import streamlit as st

print("Streamlit version:", st.__version__)

In [ ]:
%%writefile app.py
import os
import joblib
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt

st.set_page_config(
    page_title="Explainable Hybrid Intrusion Detection",
    page_icon="🛡️",
    layout="wide"
)

st.title("🛡️ Explainable Hybrid Intrusion Detection System")
st.write("XGBoost-based network intrusion detection with SHAP explainability.")

ARTIFACT_DIR = "ids_artifacts"

@st.cache_resource
def load_artifacts():
    model = joblib.load(
        os.path.join(ARTIFACT_DIR, "xgboost_multiclass.pkl")
    )
    encoder = joblib.load(
        os.path.join(ARTIFACT_DIR, "label_encoder.pkl")
    )
    features = joblib.load(
        os.path.join(ARTIFACT_DIR, "selected_features.pkl")
    )
    return model, encoder, features

try:
    model, label_encoder, selected_features = load_artifacts()
except Exception as e:
    st.error(f"Could not load model artifacts: {e}")
    st.stop()

st.success(
    f"Model loaded: {len(selected_features)} features, "
    f"{len(label_encoder.classes_)} classes"
)

st.header("1. Global SHAP Feature Importance")

importance_path = os.path.join(
    ARTIFACT_DIR, "shap_feature_importance.csv"
)

if os.path.exists(importance_path):
    importance = pd.read_csv(importance_path).head(15)
    importance = importance.sort_values("Mean_Absolute_SHAP")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(
        importance["Feature"],
        importance["Mean_Absolute_SHAP"]
    )
    ax.set_xlabel("Mean Absolute SHAP Value")
    ax.set_title("Top 15 Important Features")
    fig.tight_layout()
    st.pyplot(fig)
    plt.close(fig)

st.header("2. Network Traffic Classification")

uploaded_file = st.file_uploader(
    "Upload a CSV containing the 35 selected features",
    type=["csv"]
)

if uploaded_file is not None:
    data = pd.read_csv(uploaded_file)

    missing = [
        feature for feature in selected_features
        if feature not in data.columns
    ]

    if missing:
        st.error(f"Missing required features: {missing}")
    else:
        X = data[selected_features].apply(
            pd.to_numeric, errors="coerce"
        )

        valid_mask = X.notna().all(axis=1)

        if (~valid_mask).any():
            st.warning(
                f"{(~valid_mask).sum()} rows have missing or "
                "non-numeric values and will be skipped."
            )

        X_valid = X.loc[valid_mask]

        if X_valid.empty:
            st.error("No valid rows to classify.")
        else:
            predictions = model.predict(X_valid)
            probabilities = model.predict_proba(X_valid)

            predicted_labels = label_encoder.inverse_transform(
                predictions.astype(int)
            )

            results = data.loc[X_valid.index].copy()
            results["Predicted_Class"] = predicted_labels
            results["Confidence_Percent"] = (
                probabilities.max(axis=1) * 100
            ).round(2)

            st.subheader("Prediction Results")
            st.dataframe(results, use_container_width=True)

            st.download_button(
                "Download Prediction Results",
                results.to_csv(index=False).encode("utf-8"),
                file_name="intrusion_detection_results.csv",
                mime="text/csv"
            )

            st.subheader("Prediction Distribution")
            st.bar_chart(results["Predicted_Class"].value_counts())

            st.caption(
                "Confidence represents the model's predicted "
                "probability, not a guarantee of correctness."
            )

st.header("3. Detection Status")
st.info(
    "The current version uses XGBoost classification and global "
    "SHAP feature importance. Validated rule-based detection "
    "has not yet been integrated."
)

In [ ]:
import os

print("Current directory:", os.getcwd())
print("Artifact folder exists:", os.path.isdir("ids_artifacts"))

if os.path.isdir("ids_artifacts"):
    print(os.listdir("ids_artifacts"))

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

In [ ]:
import subprocess
import time
import os

# Start Streamlit without using shell background syntax
log_file = open("streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit process running:", process.poll() is None)

if process.poll() is not None:
    log_file.flush()
    with open("streamlit.log", "r") as f:
        print(f.read())

In [ ]:
with open("streamlit.log", "r") as f:
    print(f.read())

In [ ]:
!pip -q install pyngrok

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("Open your Streamlit dashboard here:")
print(public_url)